# Astra ByT5 기존 가중치 재사용 — G4

**새 학습0스텝, 대회 제출0회.** 이전 fold2/fold4 모델을 Drive에서 찾아 카탈로그 전체 검색을 검증합니다. 이전 학습 폴더를 그대로 보관하세요.

위부터 모두 실행하고, 업로드 창에 `Astra_ByT5_Reuse_G4_20261031.zip`을 넣으세요. 끝나면 `REUSE_RESULTS_RETURN_TO_ASTRA.zip`만 전달하면 됩니다. 새 제출ZIP은 만들지 않습니다.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import sys, subprocess, json, zipfile, datetime
drive.mount('/content/drive')
subprocess.run(['nvidia-smi'], check=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.6', 'lightgbm==4.6.0', 'rapidfuzz', 'pandas', 'numpy<3', 'scipy'])


In [ ]:
uploaded = files.upload()
names = [n for n in uploaded if n.endswith('.zip')]
assert len(names) == 1, '재사용 실행 묶음 ZIP 하나를 선택하세요.'
stamp = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
work = Path('/content') / ('astra_byt5_reuse_' + stamp)
work.mkdir(exist_ok=False)
with zipfile.ZipFile(names[0]) as archive:
    for n in archive.namelist():
        assert not Path(n).is_absolute() and '..' not in Path(n).parts
    archive.extractall(work)
package = work / 'astra_reuse'
previous_root = Path('/content/drive/MyDrive/astra_byt5_20261031')
run_dir = Path('/content/drive/MyDrive/astra_byt5_reuse_20261031') / ('run_' + stamp)
assert previous_root.is_dir(), '이전 ByT5 실행 결과가 있는 Drive 계정인지 확인하세요.'
del uploaded
print('기존 모델 탐색:', previous_root)
print('새 검증 결과:', run_dir)


In [ ]:
proc = subprocess.Popen([sys.executable, '-u', str(package/'run_reuse.py'), str(previous_root), str(run_dir)],
                        cwd='/tmp', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
print('process returncode:', proc.wait())


In [ ]:
status_path = run_dir / 'FINAL_STATUS.json'
if status_path.exists():
    result = json.loads(status_path.read_text())
    print('상태:', result['status'])
    for fold, report in result.get('reports', {}).items():
        print(fold, '기존 누락 정답 추가검색:', report['newly_recalled'], '/', report['original_missing_hard'])
        for name, metric in report['methods'].items():
            print(name, 'clean 훼손=', metric['clean']['harm'], 'noise 순증=', metric['noise']['net'])
return_zip = run_dir / 'REUSE_RESULTS_RETURN_TO_ASTRA.zip'
assert return_zip.exists(), f'로그와 {run_dir}를 확인하세요.'
files.download(str(return_zip))
